# Utils - QB - DurationNormalizer

Ce notebook illustre et vérifie le comportement de la classe `DurationNormalizer`
(`tsforecast/utils/duration/normalizer.py`), qui centralise la normalisation des
représentations de durées (codes pandas-like `'D'`, `'M'`, `'Q'`... vs noms
littéraux `'day'`, `'month'`, `'quarter'`...). Cette classe est utilisée en interne
par `DurationConverter` (voir `duration_converter.ipynb`) ainsi que par les
fonctions de commodité de `tsforecast/utils/duration/utils.py`.

On se concentre ici sur les **méthodes publiques** de `DurationNormalizer`
(`normalize`, `to_code`, `to_literal`, `validate`, `is_longer_duration`,
`are_compatible_durations`) ainsi que sur les **fonctions publiques** de
`utils.py` qui les exposent au niveau module (`normalize_duration`, `to_code`,
`to_literal`, `validate_duration`, `get_duration_order`). Les méthodes/fonctions
privées (préfixées `_`) et la classe abstraite parente `TemporalNormalizer` ne
sont pas testées directement.

## 1 - Import et instanciation

In [ ]:
# Importation des modules
from typing import get_args

# Classe testée
from tsforecast.utils.duration.normalizer import DurationNormalizer

# Types exportés (pour lister les unités supportées, à titre indicatif)
from tsforecast.utils.duration.types import DurationType, UserDurationType

# Instanciation du normalizer
normalizer = DurationNormalizer()

codes = list(get_args(DurationType))
litteraux = list(get_args(UserDurationType))

print("Codes de durée déclarés dans DurationType :")
print(codes)
print()
print("Noms littéraux déclarés dans UserDurationType :")
print(litteraux)

## 2 - `normalize()` : normalisation vers un code

### 2.1 - Codes et littéraux valides

In [ ]:
# Un code est toujours renvoyé inchangé
for c in codes:
    assert normalizer.normalize(c) == c, c
print("OK : normalize(code) == code pour tous les codes déclarés")

# Chaque nom littéral est converti vers le code correspondant
# (la correspondance code <-> littéral suit l'ordre de codes/litteraux ci-dessus)
for c, lit in zip(codes, litteraux):
    resultat = normalizer.normalize(lit)
    print(f"{lit:15s} -> {resultat}")
    assert resultat == c, (lit, resultat, c)

### 2.2 - Sensibilité à la casse (point de vigilance)

`normalize()` fait un lookup exact dans des dictionnaires (`_code_to_literal` /
`_literal_to_code`) : aucune normalisation de casse n'est appliquée en amont.
Une variante mal capitalisée d'un code ou d'un littéral pourtant valide est donc
rejetée.

In [ ]:
# 'D' (code valide) fonctionne, mais 'd' (minuscule) échoue
print("'D' ->", normalizer.normalize('D'))
try:
    normalizer.normalize('d')
except ValueError as e:
    print("'d' -> ValueError :", e)

# Idem pour les littéraux : 'hour' fonctionne, 'HOUR' et 'Hour' échouent
print("'hour' ->", normalizer.normalize('hour'))
for variante in ['HOUR', 'Hour']:
    try:
        normalizer.normalize(variante)
        print(f"{variante!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{variante!r} -> ValueError : {e}")

# 'min' (minute) est un code valide, 'Min' ne l'est pas
print("'min' ->", normalizer.normalize('min'))
try:
    normalizer.normalize('Min')
except ValueError as e:
    print("'Min' -> ValueError :", e)

### 2.3 - Repli sur `parse_frequency` pour les chaînes de fréquence pandas

Si la valeur n'est trouvée ni comme code ni comme littéral, `normalize()` tente
d'extraire une fréquence de base via `parse_frequency()` (`tsforecast/utils/parse/`),
qui décompose une chaîne `[FREQ][S|E]?[-SUFFIXE]?`. Cela permet d'accepter des
chaînes de fréquence pandas complètes (`'MS'`, `'QE-DEC'`...) en ignorant la
position (start/end) et le suffixe d'ancrage.

In [ ]:
# La position (S/E) et le suffixe d'ancrage sont ignorés : seule la base compte
exemples = ['MS', 'ME', 'QE-DEC', 'QS-JAN', 'YE-DEC', 'W-MON']
for ex in exemples:
    resultat = normalizer.normalize(ex)
    print(f"{ex:10s} -> {resultat}")

assert normalizer.normalize('MS') == normalizer.normalize('ME') == 'M'
assert normalizer.normalize('QE-DEC') == normalizer.normalize('QS-JAN') == 'Q'
print("OK : la position et le suffixe n'affectent pas le code renvoyé")

#### Piège : `'S'` (fréquence pandas historique des secondes) n'est PAS reconnu

Le code de durée pour la seconde est `'s'` (minuscule). La chaîne `'S'` seule
(majuscule) n'est ni un code ni un littéral connu ; le repli via
`parse_frequency('S')` extrait une base identique à la valeur d'entrée (`'S'`),
ce qui empêche toute récursion (garde-fou anti boucle infinie) et lève donc une
erreur, au lieu de retomber sur la seconde.

In [ ]:
print("'s' (minuscule, code valide) ->", normalizer.normalize('s'))
try:
    normalizer.normalize('S')
except ValueError as e:
    print("'S' (majuscule) -> ValueError :", e)

### 2.4 - Erreurs : types non `str` et chaînes non supportées

Contrairement à `DurationConverter.convert()` (qui laisse remonter un `TypeError`
natif via `value * facteur`), `DurationNormalizer.normalize()` vérifie
explicitement le type de l'entrée et lève toujours un `ValueError`, y compris
pour des types manifestement invalides comme `None`, un entier ou une liste.

In [ ]:
for valeur_invalide in [5, None, ['D'], 3.14]:
    try:
        normalizer.normalize(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

# Chaîne str mais non reconnue, même après repli sur parse_frequency
for valeur_invalide in ['xyz', '']:
    try:
        normalizer.normalize(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

## 3 - `to_code()` : alias explicite de `normalize()`

`to_code()` ne fait qu'appeler `normalize()` en interne : les deux méthodes sont
strictement équivalentes sur toutes les entrées (valides ou non).

In [ ]:
for valeur in codes + litteraux + ['MS', 'xyz']:
    try:
        r_normalize = normalizer.normalize(valeur)
    except ValueError as e:
        r_normalize = f'ValueError: {e}'
    try:
        r_to_code = normalizer.to_code(valeur)
    except ValueError as e:
        r_to_code = f'ValueError: {e}'
    assert r_normalize == r_to_code, (valeur, r_normalize, r_to_code)
print("OK : to_code(x) == normalize(x) pour toutes les valeurs testées (y compris les erreurs)")

## 4 - `to_literal()` : conversion vers le nom littéral

### 4.1 - Comportement de base et bijection code <-> littéral

In [ ]:
# Conversion code -> littéral et littéral -> littéral (idempotent : normalize puis lookup)
for c, lit in zip(codes, litteraux):
    assert normalizer.to_literal(c) == lit
    assert normalizer.to_literal(lit) == lit
print("OK : to_literal(code) == to_literal(littéral) == littéral, pour toutes les paires")

# Round-trip complet : to_code(to_literal(code)) == code
for c in codes:
    assert normalizer.to_code(normalizer.to_literal(c)) == c
print("OK : to_code(to_literal(code)) == code (bijection code <-> littéral)")

### 4.2 - Erreurs

In [ ]:
# to_literal() délègue à normalize() : mêmes erreurs (ValueError), y compris types non str
for valeur_invalide in ['xyz', None, 5]:
    try:
        normalizer.to_literal(valeur_invalide)
        print(f"{valeur_invalide!r} : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"{valeur_invalide!r} -> ValueError : {e}")

## 5 - `validate()` : vérification booléenne, ne lève jamais d'exception

`validate()` encapsule `normalize()` dans un `try/except ValueError` : quelle que
soit l'entrée (chaîne inconnue, type non `str`, valeur `None`...), elle renvoie
toujours un booléen sans jamais propager d'exception.

In [ ]:
# Valeurs valides
for valeur in ['day', 'D', 'quarter', 'MS', 'QE-DEC']:
    print(f"validate({valeur!r}) = {normalizer.validate(valeur)}")

print()

# Valeurs invalides de tous types : aucune exception ne remonte, toujours False
for valeur_invalide in ['xyz', '', 'd', 'HOUR', None, 5, 3.14, ['D'], {'a': 1}]:
    resultat = normalizer.validate(valeur_invalide)
    print(f"validate({valeur_invalide!r}) = {resultat}")
    assert resultat is False

## 6 - `is_longer_duration()` : comparaison stricte de deux durées

### 6.1 - Comportement de base et formats mixtes (code / littéral)

In [ ]:
# Comparaisons simples, avec formats mixtes
print("month > day  :", normalizer.is_longer_duration('month', 'day'))
print("M > D        :", normalizer.is_longer_duration('M', 'D'))
print("month > D    :", normalizer.is_longer_duration('month', 'D'))
print("day > month  :", normalizer.is_longer_duration('day', 'month'))
print("week > quarter :", normalizer.is_longer_duration('week', 'quarter'))

### 6.2 - Égalité : toujours `False` (comparaison stricte `>`, pas `>=`)

In [ ]:
# is_longer_duration(x, x) est toujours False : la comparaison est stricte
for c in codes:
    assert normalizer.is_longer_duration(c, c) is False
print("OK : is_longer_duration(x, x) == False pour tous les codes (comparaison stricte)")

### 6.3 - Rang des durées "spéciales" : `B` (jour ouvré) et `SM` (semi-mois)

`B` (business_day) et `SM` (semi_month) ont un ordre non entier (`7.5` et `8.5`
respectivement dans `_duration_order`), ce qui les place strictement entre deux
durées standards : `D < B < W` et `W < SM < M`.

In [ ]:
assert normalizer.is_longer_duration('B', 'D') is True   # jour ouvré > jour
assert normalizer.is_longer_duration('W', 'B') is True   # semaine > jour ouvré
assert normalizer.is_longer_duration('SM', 'W') is True  # semi-mois > semaine
assert normalizer.is_longer_duration('M', 'SM') is True  # mois > semi-mois
print("OK : D < B < W < SM < M")

### 6.4 - Matrice complète d'ordre (utile comme table de référence)

In [ ]:
import pandas as pd

matrice = pd.DataFrame(
    {b: [normalizer.is_longer_duration(a, b) for a in codes] for b in codes},
    index=codes
)
matrice.index.name = 'a \\ b (a > b ?)'
matrice

### 6.5 - Erreur propagée si une durée est invalide (pas de repli silencieux sur 0)

`_duration_order` utilise `.get(code, 0)` en interne, ce qui pourrait laisser
penser qu'une durée inconnue est simplement traitée comme "la plus courte"
(ordre `0`). En réalité, `is_longer_duration()` appelle d'abord `to_code()` sur
les deux arguments, qui lève un `ValueError` **avant** d'atteindre ce `.get()` :
le repli à `0` est donc du code mort avec les mappings actuels (tous les codes
déclarés ont une entrée dans `_duration_order`).

In [ ]:
for a, b in [('day', 'xyz'), ('xyz', 'day')]:
    try:
        normalizer.is_longer_duration(a, b)
        print(f"is_longer_duration({a!r}, {b!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"is_longer_duration({a!r}, {b!r}) -> ValueError : {e}")

## 7 - `are_compatible_durations()` : à ne pas confondre avec une compatibilité de conversion

Le nom de la méthode suggère une vérification de compatibilité *entre* deux
durées (par exemple : est-ce que convertir `dur1` en `dur2` a du sens ?). En
pratique, l'implémentation se contente de normaliser `dur1` puis `dur2`
indépendamment, sans jamais comparer les deux résultats entre eux : elle est
donc strictement équivalente à `validate(dur1) and validate(dur2)`.

In [ ]:
# Deux durées aux ordres de grandeur radicalement différents sont jugées "compatibles"
# (aucune notion de rapport de conversion raisonnable n'est vérifiée)
print("ns / Y compatibles :", normalizer.are_compatible_durations('ns', 'Y'))
print("business_day / month compatibles :", normalizer.are_compatible_durations('business_day', 'month'))

# Équivalence stricte avec validate(dur1) and validate(dur2)
paires = [('day', 'month'), ('ns', 'Y'), ('day', 'xyz'), ('xyz', 'day'), ('xyz', 'abc')]
for a, b in paires:
    attendu = normalizer.validate(a) and normalizer.validate(b)
    obtenu = normalizer.are_compatible_durations(a, b)
    print(f"are_compatible_durations({a!r}, {b!r}) = {obtenu}  (== validate(a) and validate(b) : {attendu})")
    assert obtenu == attendu
print("OK : are_compatible_durations(a, b) == validate(a) and validate(b), sans logique conjointe supplémentaire")

## 8 - Fonctions de commodité (`tsforecast/utils/duration/utils.py`)

Ce module expose une instance globale partagée `_normalizer = DurationNormalizer()`
et des fonctions au niveau module qui délèguent directement à cette instance :
`normalize_duration`, `to_code`, `to_literal`, `validate_duration`, ainsi que
`get_duration_order` (qui accède directement à `_normalizer._duration_order`).

### 8.1 - Délégation directe : équivalence avec une instance locale de `DurationNormalizer`

In [ ]:
from tsforecast.utils.duration.utils import (
    normalize_duration, to_code as fn_to_code, to_literal as fn_to_literal,
    validate_duration, get_duration_order
)

for valeur in codes + litteraux:
    assert normalize_duration(valeur) == normalizer.normalize(valeur)
    assert fn_to_code(valeur) == normalizer.to_code(valeur)
    assert fn_to_literal(valeur) == normalizer.to_literal(valeur)
    assert validate_duration(valeur) == normalizer.validate(valeur)
print("OK : les fonctions de utils.py renvoient exactement les mêmes résultats qu'une instance dédiée de DurationNormalizer")

### 8.2 - `get_duration_order` : expose directement un attribut "privé" (`_duration_order`)

`get_duration_order()` ne passe pas par une méthode publique de `DurationNormalizer`
mais lit directement `_normalizer._duration_order.get(code, 0)` — un attribut
préfixé `_`, donc a priori interne à la classe. C'est un point de couplage à
garder en tête : toute évolution du nom ou de la structure de cet attribut dans
`DurationNormalizer` casserait silencieusement cette fonction sans qu'aucune API
publique de la classe ne le signale.

In [ ]:
for c in codes:
    print(f"{c:4s} -> ordre {get_duration_order(c)}")

# Cohérence avec is_longer_duration : get_duration_order(a) > get_duration_order(b) <=> is_longer_duration(a, b)
for a in codes:
    for b in codes:
        assert (get_duration_order(a) > get_duration_order(b)) == normalizer.is_longer_duration(a, b)
print("OK : get_duration_order est cohérent avec is_longer_duration sur toutes les paires de codes")

# Durée inconnue : repli SILENCIEUX sur 0 (contrairement à is_longer_duration, qui lève une ValueError)
# get_duration_order() appelle normalize_duration() en premier, qui lève bien une ValueError ici :
try:
    get_duration_order('xyz')
except ValueError as e:
    print("get_duration_order('xyz') -> ValueError :", e)

## 9 - Application aux indicateurs du notebook `3 - QB - Panel a frequences mixtes heterogene`

On reprend les fréquences de publication et délais définis dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (section 2) et
`duration_converter.ipynb` (section 5) : PIB trimestriel, inflation/chômage
mensuels, balance commerciale annuelle, dépenses publiques annuelles (FR/IT) ou
trimestrielles (DE).

In [ ]:
indicateurs = {
    "pib_trimestriel": {"frequence": "Q", "delai": (2, "M")},
    "inflation_ipc": {"frequence": "M", "delai": (1, "M")},
    "taux_chomage": {"frequence": "M", "delai": (1, "M")},
    "balance_commerciale_annuelle": {"frequence": "Y", "delai": (3, "M")},
    "depenses_publiques_pib (FR/IT)": {"frequence": "Y", "delai": (1, "Q")},
    "depenses_publiques_pib (DE)": {"frequence": "Q", "delai": (1, "Q")},
}

# Nom littéral de chaque fréquence de publication
for nom, infos in indicateurs.items():
    freq = infos["frequence"]
    print(f"{nom:32s} fréquence={freq} ({normalizer.to_literal(freq)})")

In [ ]:
# Le délai (exprimé dans sa propre unité) est-il plus long que la fréquence de publication ?
# Utile pour repérer les cas où le délai dépasse une période complète de la série elle-même.
for nom, infos in indicateurs.items():
    freq = infos["frequence"]
    n_delai, unite_delai = infos["delai"]
    plus_long = normalizer.is_longer_duration(unite_delai, freq)
    egal = unite_delai == freq or normalizer.to_code(unite_delai) == normalizer.to_code(freq)
    relation = "plus long que" if plus_long else ("== " if egal else "plus court que")
    print(f"{nom:32s} délai en {unite_delai} ({normalizer.to_literal(unite_delai)}), {relation} la fréquence {freq}")

## 10 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/duration/test_normalizer.py` :

- **Identité** : `normalize(code) == code` et `to_code(code) == code` pour tous les
  codes déclarés dans `DurationType`.
- **Littéraux -> codes** : `normalize(littéral)` renvoie le code correspondant, pour
  tous les littéraux de `UserDurationType`.
- **Bijection** : `to_code(to_literal(code)) == code` pour tous les codes.
- **`to_code()` est un alias strict de `normalize()`** (mêmes résultats, mêmes erreurs).
- **Sensibilité à la casse** : aucune normalisation de casse n'est appliquée — `'d'`,
  `'HOUR'`, `'Min'` sont rejetés bien que `'D'`, `'hour'`, `'min'` soient valides.
- **Repli `parse_frequency`** : les chaînes de fréquence pandas complètes (`'MS'`,
  `'QE-DEC'`, `'W-MON'`...) sont acceptées ; la position (S/E) et le suffixe
  d'ancrage sont ignorés, seule la base compte.
- **Piège `'S'` vs `'s'`** : `'S'` (majuscule, alias pandas historique des secondes)
  n'est PAS reconnu — seul `'s'` (minuscule) est un code valide ; le repli via
  `parse_frequency` échoue car la base extraite égale la valeur d'entrée (garde-fou
  anti-boucle), sans jamais retomber sur la seconde.
- **Types non `str`** : `normalize()`/`to_code()`/`to_literal()` lèvent toujours
  `ValueError` (pas `TypeError`), y compris pour `None`, un `int`, une `list`.
- **`validate()` ne lève jamais d'exception** : renvoie `True`/`False` pour
  n'importe quel type d'entrée, y compris des types non hashables comme un `dict`.
- **`is_longer_duration()` est une comparaison stricte** : toujours `False` pour
  `is_longer_duration(x, x)`.
- **Ordre des durées "spéciales"** : `D < B < W < SM < M` (`B` et `SM` ont un ordre
  non entier, `7.5` et `8.5`, pour s'insérer strictement entre deux durées standards).
- **`is_longer_duration()` propage l'erreur sur durée invalide** : le repli
  `_duration_order.get(code, 0)` est du code mort avec les mappings actuels, car
  `to_code()` lève déjà une `ValueError` avant d'atteindre ce `.get()`.
- **`are_compatible_durations()` ne vérifie rien de conjoint** : strictement
  équivalent à `validate(dur1) and validate(dur2)` — aucune notion de rapport de
  conversion raisonnable entre les deux durées n'est testée (ex. `'ns'`/`'Y'` sont
  jugées "compatibles").
- **Fonctions de `utils.py`** : délèguent à une instance globale partagée de
  `DurationNormalizer` et renvoient des résultats identiques à une instance locale.
- **`get_duration_order()`** : accède directement à l'attribut privé
  `_normalizer._duration_order` plutôt qu'à une méthode publique de
  `DurationNormalizer` — point de couplage fragile à surveiller si la classe évolue.